# Merge and extraction of Mission Gate videos

In [ ]:
#!/usr/bin/env python3
"""
Merge multiple CSV files and summarize frame ranges by seq and cam_view.

Requirements:
- Python 3
- pandas (`pip install pandas`)
"""

import pandas as pd
import glob
import os

# -------------------- USER SETTINGS --------------------
INPUT_CSV_FOLDER = "../data/csv_logs"  # Folder containing multiple CSV files
OUTPUT_CSV_FILE = "../data/csv_logs/merged_summary.csv" # Output CSV file
FILTER_SEQ = None       # e.g., "seq1" or None for all
FILTER_CAM_VIEW = None  # e.g., "cam1" or None for all
# -------------------------------------------------------

# Step 1: Read all CSV files
all_files = glob.glob(os.path.join(INPUT_CSV_FOLDER, "*.csv"))
if not all_files:
    print("No CSV files found in folder.")
    exit(1)

df_list = [pd.read_csv(f) for f in all_files]
df = pd.concat(df_list, ignore_index=True)

# Step 2: Filter seq and cam_view if specified
if FILTER_SEQ:
    df = df[df["seq"] == FILTER_SEQ]
if FILTER_CAM_VIEW:
    df = df[df["cam_view"] == FILTER_CAM_VIEW]

# Step 3: Group by seq and cam_view
grouped = df.groupby(["seq", "cam_view"])

summary_rows = []

for (seq, cam_view), group in grouped:
    # Determine min and max frame_num
    start_frame = group["frame_num"].min()
    end_frame = group["frame_num"].max()
    
    # Get first URL (assuming all rows in group are same video)
    url = group["url"].iloc[0]
    
    # Transfer additional info (assuming consistent within group)
    gait_event = group["gait_event"].iloc[0] if "gait_event" in group else ""
    dataset = group["dataset"].iloc[0] if "dataset" in group else ""
    gait_pat = group["gait_pat"].iloc[0] if "gait_pat" in group else ""
    
    summary_rows.append({
        "seq": seq,
        "cam_view": cam_view,
        "start_frame": start_frame,
        "end_frame": end_frame,
        "url": url,
        "gait_event": gait_event,
        "dataset": dataset,
        "gait_pat": gait_pat
    })

# Step 4: Save to new CSV
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_CSV_FILE, index=False)
print(f"Summary CSV saved to {OUTPUT_CSV_FILE}")


# Final Downloader - Ignores duplicates and filters for Target Uploader

In [10]:
#!/usr/bin/env python3
"""
YouTube Segment Downloader with incremental CSV enrichment
and uploader filtering. Avoids duplicates on re-runs.
"""

from yt_dlp import YoutubeDL
import os
import subprocess
import pandas as pd
import re
import hashlib
import json
import csv
import time
import random


# -------------------- USER SETTINGS --------------------
INPUT_CSV = "../data/GAVD_data/csv_logs/merged_summary_400-1900.csv"
OUTPUT_CSV = "../data/GAVD_data/MissionGate/merged_summary_enriched_2.csv"
TEMP_FOLDER = "../data/GAVD_data/MissionGate/temp_videos_2"
OUTPUT_VIDEO_FOLDER = "../data/GAVD_data/MissionGate/video_snippets_2"
TARGET_UPLOADER = "Mission Gait"  # Only download/process videos from this uploader
# ------------------------------------------------------

os.makedirs(TEMP_FOLDER, exist_ok=True)
os.makedirs(OUTPUT_VIDEO_FOLDER, exist_ok=True)

# -------------------- Utilities ----------------------

def safe_name(s):
    return re.sub(r"[^\w\-_. ]", "_", str(s)).strip()

def frame_to_timestamp(frame, fps):
    seconds = frame / fps
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = seconds % 60
    return f"{h:02}:{m:02}:{s:06.3f}"

def sha256_checksum(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

def ffprobe_metadata(file_path):
    cmd = [
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration:stream=codec_type,codec_name,width,height,r_frame_rate",
        "-print_format", "json",
        file_path
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"ffprobe failed on {file_path}")
    return json.loads(result.stdout)

"""
def download_full_video(url):
    ydl_opts = {
        "format": "bv*/b",  # best video or best combined
        "outtmpl": os.path.join(TEMP_FOLDER, "%(title)s.%(ext)s"),
        "noplaylist": True,
        "quiet": False,
        "remote_components": ["ejs:github"],
    }
    with YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=True)

    if info is None:
        raise RuntimeError("yt-dlp returned no info")
    if info.get("vcodec") == "none":
        raise RuntimeError("Audio-only stream — no video available")

    title = safe_name(info["title"])
    ext = info.get("ext")
    input_path = os.path.join(TEMP_FOLDER, f"{title}.{ext}")

    if not os.path.exists(input_path):
        raise FileNotFoundError(f"Downloaded file missing: {input_path}")

    return info, input_path
"""

def get_video_info(url):
    ydl_opts = {
        "quiet": True,
        "noplaylist": True,
        "skip_download": True,
        "cookiesfrombrowser": ("chrome",),

    }
    with YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=False)

    if info is None:
        raise RuntimeError("yt-dlp returned no info")

    return info


def download_video_with_info(info):
    ydl_opts = {
        "format": "bv*/b",
        "outtmpl": os.path.join(TEMP_FOLDER, "%(title)s.%(ext)s"),
        "noplaylist": True,
        "quiet": False,
        "remote_components": ["ejs:github"],
        "cookiesfrombrowser": ("chrome",),
        "sanitize_filename": True,  # optional but safe

    }
    with YoutubeDL(ydl_opts) as ydl:
        ydl.process_info(info)
        input_path = ydl.prepare_filename(info)  # actual filename

    #Removed and replaced with the line above as it caused matching issues on second/third run with rows 400 onwards
    """
    title = safe_name(info["title"])
    ext = info.get("ext")
    input_path = os.path.join(TEMP_FOLDER, f"{title}.{ext}")"""

    if not os.path.exists(input_path):
        raise FileNotFoundError(f"Downloaded file missing: {input_path}")

    return input_path

    
def cut_and_reencode(input_file, start_ts, end_ts, output_file):
    if not os.path.exists(output_file):
        subprocess.run([
            "ffmpeg", "-y",
            "-i", input_file,
            "-ss", start_ts,
            "-to", end_ts,
            "-c:v", "libx264",
            "-c:a", "aac",
            output_file
        ], check=True)

def append_row_to_csv(row_dict, csv_file):
    file_exists = os.path.exists(csv_file)
    with open(csv_file, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=row_dict.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(row_dict)

video_info_cache = {}

# -------------------- Main ---------------------------

def main():
    df = pd.read_csv(INPUT_CSV)

    # Load already processed rows to avoid duplicates
    if os.path.exists(OUTPUT_CSV):
        df_existing = pd.read_csv(OUTPUT_CSV)
        processed_keys = set(
            zip(df_existing["url"], df_existing["start_frame"], df_existing["end_frame"])
        )
    else:
        processed_keys = set()

    # Add enrichment columns if missing
    for col in ["title","uploader","fps","start_time","end_time","duration","checksum","width","height"]:
        if col not in df.columns:
            df[col] = ""

    for idx, row in df.iterrows():
        try:
            

            print(f"\n▶ Processing row {idx}")

            url = row["url"]
            start_frame = int(row["start_frame"])
            end_frame = int(row["end_frame"])

            output_name = safe_name(f"{row['seq']}_{row['cam_view']}_{row['gait_event']}_{row['dataset']}_{row['gait_pat']}.mp4")
            output_path = os.path.join(OUTPUT_VIDEO_FOLDER, output_name)

            # Download video
            """info, input_video = download_full_video(url)
            uploader = info.get("uploader","")
            if uploader != TARGET_UPLOADER:
                print(f"Skipping video '{info.get('title','')}' (Uploader: {uploader})")
                continue"""
            
            # Get metadata ONLY (no download yet)
            info = get_video_info(url)
            uploader = info.get("uploader", "")

            # Skip before download if uploader does not match
            if uploader != TARGET_UPLOADER:
                print(f"Skipping video '{info.get('title','')}' (Uploader: {uploader})")
                continue
            
            #Ensure no duplicated processing of URL frame segments
            key = (row["url"], row["start_frame"], row["end_frame"])
            if key in processed_keys:
                print(f"▶ Row {idx} already processed, skipping")
                continue

            # Download only approved uploader videos
            input_video = download_video_with_info(info)


            # Cut clip
            fps_list = [f.get("fps") for f in info.get("formats", []) if f.get("fps")]
            fps = max(fps_list) if fps_list else 30
            start_ts = frame_to_timestamp(start_frame, fps)
            end_ts = frame_to_timestamp(end_frame, fps)
            cut_and_reencode(input_video, start_ts, end_ts, output_path)

            if os.path.exists(input_video):
                os.remove(input_video)

            # Extract metadata
            meta = ffprobe_metadata(output_path)
            video_stream = next((s for s in meta["streams"] if s["codec_type"]=="video"), None)
            width = height = ""
            if video_stream:
                width = video_stream.get("width","")
                height = video_stream.get("height","")
                if "r_frame_rate" in video_stream:
                    num, den = map(int, video_stream["r_frame_rate"].split("/"))
                    fps = num/den if den!=0 else 30

            start_ts = frame_to_timestamp(start_frame, fps)
            end_ts = frame_to_timestamp(end_frame, fps)
            duration = round((end_frame - start_frame)/fps,3)
            checksum = sha256_checksum(output_path)

            # Build row dict
            enriched_row = row.to_dict()
            enriched_row.update({
                "title": info.get("title",""),
                "uploader": uploader,
                "fps": fps,
                "start_time": start_ts,
                "end_time": end_ts,
                "duration": duration,
                "checksum": checksum,
                "width": width,
                "height": height
            })

            append_row_to_csv(enriched_row, OUTPUT_CSV)
            processed_keys.add(key)

            print(f"✅ Saved clip: {output_path}")
            time.sleep(random.uniform(0.3, 0.8))


        except Exception as e:
            print(f"❌ Row {idx} failed: {e}")

    print(f"\n✅ Finished. Enriched CSV saved to {OUTPUT_CSV}")

# -------------------- Entry Point -------------------

if __name__ == "__main__":
    main()



▶ Processing row 0
Skipping video '50 Types of Walks' (Uploader: loveliveserve)

▶ Processing row 1
Skipping video '50 Types of Walks' (Uploader: loveliveserve)

▶ Processing row 2
Skipping video '50 Types of Walks' (Uploader: loveliveserve)

▶ Processing row 3
Skipping video '50 Types of Walks' (Uploader: loveliveserve)

▶ Processing row 4


Skipping video '50 Types of Walks' (Uploader: loveliveserve)

▶ Processing row 5


Skipping video '50 Types of Walks' (Uploader: loveliveserve)

▶ Processing row 6


Skipping video '50 Types of Walks' (Uploader: loveliveserve)

▶ Processing row 7


Skipping video '50 Types of Walks' (Uploader: loveliveserve)

▶ Processing row 8


...

Skipping video 'Gaits Examination (Stanford Medicine 25)' (Uploader: Stanford Medicine 25)

▶ Processing row 1471


Skipping video 'Gaits Examination (Stanford Medicine 25)' (Uploader: Stanford Medicine 25)

▶ Processing row 1472


Skipping video 'Gaits Examination (Stanford Medicine 25)' (Uploader: Stanford Medicine 25)

▶ Processing row 1473


Skipping video 'Weak Quadriceps Gait | Compensations for a Buckling Knee' (Uploader: ABCs of PT)

▶ Processing row 1474


Skipping video 'Weak Quadriceps Gait | Compensations for a Buckling Knee' (Uploader: ABCs of PT)

✅ Finished. Enriched CSV saved to ../data/GAVD_data/MissionGate/merged_summary_enriched_2.csv
